### **INFO-6151-(01)-26W Data Visiualization for Machine Learning**
## **Capstone Project: Impact of Geopolitical Conflicts and Trade Policies on US Market Sectors**
**Date: March 28th, 2026**

**Group Members:**

*   Yun-Jiung Wang 1256222
*   Andrew Polk 1161715
*   Ebenebe, Kaosicho 1252551

<br>

### **1. Project Objective:**

The primary objective of this project is to analyze and visualize the resilience of different US industrial sectors under the dual pressure of Global Tariff Policies and the 2026 Iranian Conflict. By utilizing the yfinance library, we aim to evaluate how specific Exchange Traded Funds (ETFs) and market sentiment indicators respond to macroeconomic shocks and geopolitical instability.

<br>

### **2. Research Scope & Methodology:**

We focus on five key assets and one sentiment benchmark representing diverse segments of the global economy:
<br>

**SPY:** S&P 500 (Broad Market Benchmark)

**QQQ:** Nasdaq 100 (Technology & Innovation)

**ITA:** Aerospace & Defense (National Security)

**ICLN:** Clean Energy (Future Sustainability)

**VDE:** Vanguard Energy (Traditional Oil & Gas)

**VIX:** CBOE Volatility Index (Market Sentiment & "Fear Gauge")

<br>

**Methodology:**

**Data Acquisition:**
Extracting real-time and historical price data via Python.


**Event Study:**
Mapping specific dates of tariff announcements and military escalations against market performance.

**Sentiment Correlation:** Comparing sector price movements against the VIX to identify "safe-haven" characteristics during periods of high fear.

 <br>

### **3. Motivation:**

**Sector Divergence:** In times of war, Defense (ITA) and Energy (VDE) often act as "hedges," while Tech (QQQ) may suffer due to supply chain disruptions caused by tariffs.

**Sentiment Analysis:** By integrating the VIX, we can visualize how global panic correlates with capital flight from or into specific industrial sectors.

**Strategic Insights:** This study provides a "stress test" for investment portfolios, identifying which industries provide stability during global crises.

<br>

### **4. Expected Technical Outcomes:**

**Interactive Time-Series Plots:** Highlighting performance "break points" and VIX spikes during key geopolitical events.

**Risk Assessment:**  Calculating maximum drawdowns, rolling volatility, and the "Fear-Performance" ratio for each sector.

**Correlation Matrix:**  Analyzing the relationship between traditional assets and the VIX during the 2026 conflict to evaluate hedging effectiveness.


In [42]:
!pip install dash
!pip install yfinance

## **Import Libs**

In [43]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
from dash import Dash, dcc, html, Input, Output
from google.colab import output

## **Data Collection:**

1. Define required tickers
2. download the raw data
3. display first 2 rows

In [44]:
# Define the Stocks for observation
STOCKS = {
    "SPY": "Market",
    "QQQ": "Tech",
    "ITA": "Defense",
    "ICLN": "Clean Energy",
    "VDE": "Energy"
}

# Define date region
START, END = "2024-01-01", datetime.today().strftime("%Y-%m-%d")

tickers = list(STOCKS.keys()) + ["^VIX"]

# Download the dataset
df_raw = yf.download(tickers, start=START, end=END, auto_adjust=True)

df_raw.head(2)

[*********************100%***********************]  6 of 6 completed


Price           Close                                                         \
Ticker           ICLN         ITA         QQQ         SPY         VDE   ^VIX   
Date                                                                           
2024-01-02  14.867379  123.488930  398.192139  459.991211  111.221504  13.20   
2024-01-03  14.529265  121.805519  393.978729  456.234619  112.676903  14.04   

Price            High                                      ...        Open  \
Ticker           ICLN         ITA         QQQ         SPY  ...         QQQ   
Date                                                       ...               
2024-01-02  15.128210  124.847463  401.653905  460.983912  ...  401.406636   
2024-01-03  14.616209  123.823629  396.619576  458.570335  ...  395.561257   

Price                                       Volume                    \
Ticker             SPY         VDE   ^VIX     ICLN     ITA       QQQ   
Date                                                                   
2024-01-02  459.514344  110.761409  13.22  4067800  403600  58026900   
2024-01-03  457.830680  111.259050  13.35  4564700  841100  47002800   

Price                               
Ticker            SPY     VDE ^VIX  
Date                                
2024-01-02  123623700  551000    0  
2024-01-03  103585900  482100    0  

[2 rows x 30 columns]

## **2. Data Preprocessing: Flattening MultiIndex Data**
<p>

  The initial dataset retrieved from yfinance contains a MultiIndex structure, which is complex for direct visualization. To simplify our workflow, we extracted only the adjusted closing prices and flattened the column headers. This process ensures the DataFrame is compatible with Plotly and easier for team members to perform further calculations, such as daily returns or volatility analysis.
</p>


In [45]:
# Get the Adj close data
df_final = df_raw['Close'].copy()

# Remove label name 'Ticker'
df_final.columns.name = None

# Convert Date to normal columns
df_final = df_final.reset_index()

# Rename ^VIX to VIX
df_final = df_final.rename(columns={'^VIX': 'VIX'})

# Verification:
print("Transformed DataFrame (Single Level Column):")
print(df_final.head())
print("\nColumns list:", df_final.columns.tolist())

Transformed DataFrame (Single Level Column):
        Date       ICLN         ITA         QQQ         SPY         VDE    VIX
0 2024-01-02  14.867379  123.488930  398.192139  459.991211  111.221504  13.20
1 2024-01-03  14.529265  121.805519  393.978729  456.234619  112.676903  14.04
2 2024-01-04  14.345718  121.795677  391.951141  454.764984  110.883461  14.13
3 2024-01-05  14.297416  121.913818  392.415985  455.387878  111.014938  13.35
4 2024-01-08  14.394019  120.525742  400.526489  461.889008  109.747307  13.08

Columns list: ['Date', 'ICLN', 'ITA', 'QQQ', 'SPY', 'VDE', 'VIX']


Calculate Returns and Volatility (20-day rolling)

In [46]:
# Set Date as index for time series operations
df_close = df_final.set_index("Date")

# Calculate Daily Percentage Returns
returns = df_close.pct_change()
# Computes 20-day rolling volatility annualized to a percentage
volatility = returns.rolling(window=20).std() * (252**0.5) * 100

df_raw

print("Returns shape:", returns.shape)
print("Volatility shape:", volatility.shape)

Returns shape: (561, 6)
Volatility shape: (561, 6)


Price Data for Event Analysis

In [47]:
# Separate ETF prices and VIX series for modules
prices = df_close[list(STOCKS.keys())].copy()
prices.index = pd.to_datetime(prices.index)
prices.dropna(how="all", inplace=True)

# VIX into its own series
vix = df_close["VIX"].copy()
vix.index = pd.to_datetime(vix.index)

print(f"Prices shape: {prices.shape}")
print(f"Range: {prices.index[0].date()} -> {prices.index[-1].date()}")
prices.tail(5)

Prices shape: (561, 5)
Range: 2024-01-02 -> 2026-03-27


,SPY,QQQ,ITA,ICLN,VDE
Date,,,,,
2026-03-23,655.380005,588.000000,223.350006,18.180000,169.429993
2026-03-24,653.179993,583.979980,222.639999,18.309999,172.119995
2026-03-25,656.820007,587.820007,225.860001,18.809999,171.440002
2026-03-26,645.090027,573.789978,220.119995,18.030001,174.139999
2026-03-27,634.090027,562.580017,216.039993,17.850000,176.949997


## Event Definitions

In [48]:
EVENTS = {
    # Iran War Dates:
    "Israel strikes Iran": "2026-02-28",
    # Trump Tariff Dates:
    "Tariff":"2025-04-02"
}

## Iran War Module

In [49]:
# Pull Iran date from EVENTS
IRAN_DATE = pd.Timestamp(next(v for k, v in EVENTS.items() if "Iran" in k))

# Get event context
WINDOW_START = IRAN_DATE - pd.Timedelta(days=10)
WINDOW_END   = min(IRAN_DATE + pd.Timedelta(days=30), prices.index[-1])

iran_prices = prices.loc[WINDOW_START:WINDOW_END].copy()

# Normalizes all ETF prices to 0% on the day before the strike so all lines start at the same baseline
pre_strike = prices.loc[:IRAN_DATE].iloc[-2]
iran_norm  = ((iran_prices / pre_strike) -1) * 100

# VIX series for the same window
iran_vix = vix.loc[WINDOW_START:WINDOW_END]

# Defines sector groupings
SECTORS = {
    "All":          ["SPY", "QQQ", "ITA", "ICLN", "VDE"],
    "Broad Market": ["SPY", "QQQ"],
    "Defense":      ["ITA"],
    "Energy":       ["ICLN", "VDE"],
}

# Defines ETF colours
COLORS = {
    "SPY":  "#1D9E75",
    "QQQ":  "#3C3489",
    "ITA":  "#534AB7",
    "ICLN": "#2CA02C",
    "VDE":  "#EF8B2C",
}

# Function to build the stacked subplot chart
def build_iran_fig(selected_sector):
    tickers_sel = SECTORS[selected_sector]
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.06,
        subplot_titles=["Sector % Change from Pre-Event Close", "VIX (Market Fear Index)"],
    )

    # First row: sector ETF with normalized returns
    for ticker in tickers_sel:
        if ticker not in iran_norm.columns:
            continue
        fig.add_trace(go.Scatter(
            x=iran_norm.index, y=iran_norm[ticker],
            name=f"{ticker} - {STOCKS[ticker]}",
            line=dict(color=COLORS[ticker], width=2),
            hovertemplate=f"<b>{ticker}</b><br>%{{x|%b %d %Y}}<br>%{{y:.1f}}%<extra></extra>",
        ), row=1, col=1)

    # Second row: VIX plot
    fig.add_trace(go.Scatter(
        x=iran_vix.index, y=iran_vix.values,
        name="VIX", showlegend=True,
        line=dict(color="gray", width=1.5),
        fill="tozeroy", fillcolor="rgba(180,180,180,0.15)",
        hovertemplate="<b>VIX</b><br>%{x|%b %d %Y}<br>%{y:.1f}<extra></extra>",
    ), row=2, col=1)

    # Event line + shading on both rows
    for row in [1, 2]:
        fig.add_vline(x=IRAN_DATE.timestamp()*1000,
                      line=dict(color="red", width=2, dash="dash"), row=row, col=1)
    fig.add_vrect(x0=IRAN_DATE, x1=IRAN_DATE+pd.Timedelta(days=7),
                  fillcolor="red", opacity=0.07, line_width=0)

    # Baseline on row 1
    fig.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"), row=1, col=1)

    # Event label
    visible = [t for t in tickers_sel if t in iran_norm.columns]
    fig.add_annotation(
        x=IRAN_DATE, y=iran_norm[visible].max().max(),
        text="  Israel strikes Iran",
        showarrow=False, font=dict(size=11, color="red"),
        xanchor="left", row=1, col=1,
    )

    fig.update_yaxes(title_text="% change", row=1, col=1)
    fig.update_yaxes(title_text="VIX", row=2, col=1)
    fig.update_xaxes(tickformat="%b %d", tickangle=45, row=2, col=1)
    fig.update_layout(
        title=f"Iran War - Sector: {selected_sector}",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(r=40),
        height=550,
    )
    return fig

app_iran = Dash(__name__)

app_iran.layout = html.Div([
    html.H2("Iran-Israel Conflict: Market Reaction by Sector"),
    html.P(f"% change from close on {(IRAN_DATE - pd.Timedelta(days=1)).strftime('%b %d %Y')}"),
    html.Label("View by sector:"),
    dcc.Dropdown(
        id="sector-dropdown",
        options=[{"label": k, "value": k} for k in SECTORS.keys()],
        value="All", clearable=False, style={"width": "300px"},
    ),
    dcc.Graph(id="iran-chart"),
])

@app_iran.callback(Output("iran-chart", "figure"), Input("sector-dropdown", "value"))
def update_iran_chart(selected_sector):
    return build_iran_fig(selected_sector)

#app_iran.run(port=8050, debug=False)
#output.serve_kernel_port_as_iframe(8050)

## Trump Tariffs Module

In [50]:
TARIFF_DATE = pd.Timestamp(next(v for k, v in EVENTS.items() if "Tariff" in k))

TARIFF_WINDOW_START = TARIFF_DATE - pd.Timedelta(days=30)
TARIFF_WINDOW_END   = min(TARIFF_DATE + pd.Timedelta(days=60), prices.index[-1])

tariff_prices = prices.loc[TARIFF_WINDOW_START:TARIFF_WINDOW_END].copy()

pre_tariff  = prices.loc[:TARIFF_DATE].iloc[-1]
tariff_norm = ((tariff_prices / pre_tariff) - 1) * 100

tariff_vix = vix.loc[TARIFF_WINDOW_START:TARIFF_WINDOW_END]

TARIFF_EVENTS = {
    "Liberation Day":      pd.Timestamp("2025-04-02"),
    "90-day Pause":        pd.Timestamp("2025-04-09"),
    "US-China Escalation": pd.Timestamp("2025-04-15"),
}

TARIFF_SECTORS = {
    "All":          ["SPY", "QQQ", "ITA", "ICLN", "VDE"],
    "Broad Market": ["SPY", "QQQ"],
    "Defense":      ["ITA"],
    "Energy":       ["ICLN", "VDE"],
}

def build_tariff_fig(selected_sector):
    tickers_sel = TARIFF_SECTORS[selected_sector]
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.06,
        subplot_titles=["Sector % Change from Pre-Announcement Close", "VIX (Market Fear Index)"],
    )

    # First row: sector ETF with normalized returns
    for ticker in tickers_sel:
        if ticker not in tariff_norm.columns:
            continue
        fig.add_trace(go.Scatter(
            x=tariff_norm.index, y=tariff_norm[ticker],
            name=f"{ticker} - {STOCKS[ticker]}",
            line=dict(color=COLORS[ticker], width=2),
            hovertemplate=f"<b>{ticker}</b><br>%{{x|%b %d %Y}}<br>%{{y:.1f}}%<extra></extra>",
        ), row=1, col=1)

    # Second row: VIX plot
    fig.add_trace(go.Scatter(
        x=tariff_vix.index, y=tariff_vix.values,
        name="VIX", showlegend=True,
        line=dict(color="gray", width=1.5),
        fill="tozeroy", fillcolor="rgba(180,180,180,0.15)",
        hovertemplate="<b>VIX</b><br>%{x|%b %d %Y}<br>%{y:.1f}<extra></extra>",
    ), row=2, col=1)

    # Event lines on both rows
    event_colors = {"Liberation Day": "red", "90-day Pause": "green", "US-China Escalation": "orange"}
    visible = [t for t in tickers_sel if t in tariff_norm.columns]
    y_range = tariff_norm[visible].max().max() - tariff_norm[visible].min().min() if visible else 10
    y_min   = tariff_norm[visible].min().min() if visible else -5
    y_offsets = [0.85, 0.70, 0.55]

    for (label, date), y_frac in zip(TARIFF_EVENTS.items(), y_offsets):
        color = event_colors.get(label, "gray")
        y_pos = y_min + y_frac * y_range
        for row in [1, 2]:
            fig.add_vline(x=date.timestamp()*1000,
                          line=dict(color=color, width=1.5, dash="dash"), row=row, col=1)
        fig.add_annotation(x=date, y=y_pos, text=f" {label}",
                           showarrow=False, font=dict(size=10, color=color),
                           xanchor="left", bgcolor="white", borderpad=2,
                           row=1, col=1)

    fig.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"), row=1, col=1)
    fig.update_yaxes(title_text="% change", row=1, col=1)
    fig.update_yaxes(title_text="VIX", row=2, col=1)
    fig.update_xaxes(tickformat="%b %d", tickangle=45, row=2, col=1)
    fig.update_layout(
        title=f"Trump Tariffs - Sector: {selected_sector}",
        hovermode="x unified",
        template="plotly_white",
        margin=dict(r=40),
        height=550,
    )
    return fig

app_tariff = Dash(__name__)

app_tariff.layout = html.Div([
    html.H2("Trump Tariffs (2025): Market Reaction by Sector"),
    html.P(f"% change from close on {(TARIFF_DATE - pd.Timedelta(days=1)).strftime('%b %d %Y')}"),
    html.Label("View by sector:"),
    dcc.Dropdown(
        id="tariff-sector-dropdown",
        options=[{"label": k, "value": k} for k in TARIFF_SECTORS.keys()],
        value="All", clearable=False, style={"width": "300px"},
    ),
    dcc.Graph(id="tariff-chart"),
])

@app_tariff.callback(Output("tariff-chart", "figure"), Input("tariff-sector-dropdown", "value"))
def update_tariff_chart(selected_sector):
    return build_tariff_fig(selected_sector)

#app_tariff.run(debug=True)

## Preprocessing

In [51]:
# Feature Engineering
import plotly.express as px
from sklearn.preprocessing import StandardScaler

# Log daily returns (no VIX)
log_returns = np.log(prices / prices.shift(1)).dropna()

# 20-day rolling volatility (annualized)
rolling_vol = log_returns.rolling(20).std() * np.sqrt(252)

# RSI (14-day)
def compute_rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))

rsi = prices.apply(compute_rsi)

# MACD (12/26 EMA diff)
def compute_macd(series):
    ema12 = series.ewm(span=12, adjust=False).mean()
    ema26 = series.ewm(span=26, adjust=False).mean()
    return ema12 - ema26

macd = prices.apply(compute_macd)

print("Features engineered: log_returns, rolling_vol, RSI, MACD")
print(f"  log_returns shape: {log_returns.shape}")

# Cross-Sector Correlation
# Include VIX log returns for correlation context
log_returns_vix = pd.concat([log_returns, np.log(vix / vix.shift(1)).rename("VIX")], axis=1).dropna()

snapshots_tariff = {
    "Pre-Tariff (Jan–Mar 2025)":  ("2025-01-01", "2025-04-01"),
    "Post-Tariff (Apr–Jun 2025)": ("2025-04-02", "2025-07-01"),
}
snapshots_iran = {
    "Pre-Iran (Dec 2025–Feb 2026)": ("2025-12-01", "2026-02-27"),
    "Post-Iran (Mar 2026–now)":     ("2026-02-28", prices.index[-1].strftime("%Y-%m-%d")),
}

corr_data_tariff = {}
for label, (start, end) in snapshots_tariff.items():
    w = log_returns_vix.loc[start:end]
    if len(w) > 5:
        corr_data_tariff[label] = w.corr().round(2)

corr_data_iran = {}
for label, (start, end) in snapshots_iran.items():
    w = log_returns_vix.loc[start:end]
    if len(w) > 5:
        corr_data_iran[label] = w.corr().round(2)

print("Correlation snapshots:", list(corr_data_tariff.keys()) + list(corr_data_iran.keys()))

# Before and after scaling
SCALE_TICKER = "SPY"
raw_series = log_returns[SCALE_TICKER].dropna()
scaler_std = StandardScaler()
scaled_std = scaler_std.fit_transform(raw_series.values.reshape(-1, 1)).flatten()

fig_scale = make_subplots(rows=1, cols=2, subplot_titles=["Raw Log Returns", "StandardScaler"])
for col_idx, (data, color) in enumerate([
    (raw_series.values, "steelblue"),
    (scaled_std, "darkorange"),
], start=1):
    fig_scale.add_trace(
        go.Histogram(x=data, nbinsx=60, marker_color=color, showlegend=False),
        row=1, col=col_idx,
    )
fig_scale.update_layout(
    title_text=f"{SCALE_TICKER} Log Return Distribution - Raw vs Standardized",
    template="plotly_white",
)
fig_scale.show()

Features engineered: log_returns, rolling_vol, RSI, MACD
  log_returns shape: (560, 5)
Correlation snapshots: ['Pre-Tariff (Jan–Mar 2025)', 'Post-Tariff (Apr–Jun 2025)', 'Pre-Iran (Dec 2025–Feb 2026)', 'Post-Iran (Mar 2026–now)']


## Model

In [52]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

FEATURE_TICKERS = list(STOCKS.keys())

frames = []
for tk in FEATURE_TICKERS:
    df_tk = pd.DataFrame({
        "log_ret": log_returns[tk], # todays log return
        "rolling_vol": rolling_vol[tk], # how volatile the last 20 days were
        "rsi": rsi[tk], # momentum indicator 0-100
        "macd": macd[tk], # # short vs long term trend difference
        "target": log_returns[tk].shift(-5), # what we are trying to predict, return 5 days from now
    }).dropna()
    df_tk["ticker"] = tk
    frames.append(df_tk)

model_df = pd.concat(frames).dropna()
model_df = model_df.sort_index()  # ensure chronological order

# train on 2024, test on 2025 onwards (test predictive ability)
train_df = model_df.loc[:"2024-12-31"]
test_df  = model_df.loc["2025-01-01":]

FEATURES = ["log_ret", "rolling_vol", "rsi", "macd"]

X_train = train_df[FEATURES].values
y_train = train_df["target"].values
X_test  = test_df[FEATURES].values
y_test  = test_df["target"].values

scaler_model = StandardScaler()
X_train = scaler_model.fit_transform(X_train)
X_test  = scaler_model.transform(X_test)

model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
print(f"XGBoost (walk-forward)  |  R²: {r2:.4f}  |  MSE: {mse:.6f}")

residuals = y_test - y_pred

fig_resid = make_subplots(rows=1, cols=2, subplot_titles=["Predicted vs Actual", "Residuals"])
fig_resid.add_trace(go.Scatter(x=y_test, y=y_pred, mode="markers",
    marker=dict(color="steelblue", opacity=0.5, size=4), name="Pred vs Actual"), row=1, col=1)
fig_resid.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()],
    mode="lines", line=dict(color="red", dash="dash"), name="Perfect fit"), row=1, col=1)
fig_resid.add_trace(go.Scatter(x=y_pred, y=residuals, mode="markers",
    marker=dict(color="darkorange", opacity=0.5, size=4), name="Residuals"), row=1, col=2)
fig_resid.add_hline(y=0, line=dict(color="red", dash="dash"), row=1, col=2)
fig_resid.update_layout(title="Model Diagnostics", template="plotly_white", showlegend=False)
fig_resid.show()

qq = stats.probplot(residuals, dist="norm")
qq_x = list(qq[0][0])
qq_y = list(qq[0][1])

fig_qq = go.Figure()
fig_qq.add_trace(go.Scatter(x=qq_x, y=qq_y, mode="markers",
    marker=dict(color="steelblue", size=4), name="Residuals"))
fig_qq.add_trace(go.Scatter(x=[min(qq_x), max(qq_x)],
    y=[qq[1][1]+qq[1][0]*min(qq_x), qq[1][1]+qq[1][0]*max(qq_x)],
    mode="lines", line=dict(color="red"), name="Normal line"))
fig_qq.update_layout(title="Q-Q Plot of Residuals",
    xaxis_title="Theoretical Quantiles", yaxis_title="Sample Quantiles",
    template="plotly_white")
fig_qq.show()

XGBoost (walk-forward)  |  R²: -0.1648  |  MSE: 0.000235


## SHAP

## Dash App